# Session 3: Model Evaluation and Comparison (Student Notebook)

In this session, you will learn how to systematically evaluate and compare AI model outputs using structured rubrics, automated judging, benchmarking, cost analysis, and A/B testing frameworks.

## Learning Objectives

By the end of this session, you will be able to:

1. Define and implement structured evaluation rubrics for AI responses
2. Use LLM-as-a-Judge for automated quality assessment
3. Benchmark response quality across different model configurations
4. Analyze latency and cost trade-offs between models
5. Design and run A/B tests to compare model configurations
6. Build end-to-end evaluation pipelines with reporting and visualization
7. Apply scikit-learn classification metrics (precision, recall, F1) to evaluate LLM outputs
8. Use DeepEval and G-Eval for automated, research-backed LLM scoring

In [ ]:
from dotenv import load_dotenvload_dotenv()  # Load environment variables from .envimport openaiimport jsonimport osimport timeimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

---
## Demo 1: Setting Up Evaluation Metrics and Scoring Rubrics

Before comparing models, we need a consistent framework for measuring quality. This demo shows how to define evaluation criteria and build a structured scoring rubric.

In [ ]:
client = openai.OpenAI()

# Define evaluation criteria for McKinsey consulting deliverables
evaluation_criteria = {
    "strategic_relevance": "How relevant is the analysis to the client's strategic question? (1-5)",
    "analytical_rigor": "How data-driven and methodologically sound is the analysis? (1-5)",
    "mece_structure": "How well-structured and MECE is the response? (1-5)",
    "actionability": "How actionable are the recommendations for the client? (1-5)",
    "executive_readiness": "Is this ready for a C-suite presentation? (1-5)"
}

def create_scoring_rubric():
    rubric = {}
    for criterion, description in evaluation_criteria.items():
        rubric[criterion] = {
            "description": description,
            "1": "Poor - Fails to meet basic expectations for consulting deliverables",
            "2": "Below Average - Partially addresses the criterion; not client-ready",
            "3": "Average - Adequately meets the criterion but lacks McKinsey-level polish",
            "4": "Good - Exceeds expectations; approaching partner-quality output",
            "5": "Excellent - Exceptional quality; ready for CEO/board presentation"
        }
    return rubric

rubric = create_scoring_rubric()
for criterion, details in rubric.items():
    print(f"\n{criterion.upper()}: {details['description']}")
    for score, desc in details.items():
        if score != "description":
            print(f"  {score}: {desc}")

---
## Demo 2: Automated Evaluation with LLM-as-a-Judge

Instead of manually scoring every response, we can use an LLM to act as a judge. This demo implements an automated evaluation function that scores responses against our rubric.

In [ ]:
def llm_judge(question, response_text, criteria):
    judge_prompt = f"""You are a McKinsey engagement manager evaluating AI-generated consulting analysis. Rate the following response on a scale of 1-5 for each criterion.

Client question: {question}

Response to evaluate:
{response_text}

Criteria to evaluate:
{json.dumps(criteria, indent=2)}

Return your evaluation as a JSON object with each criterion as a key and an object containing "score" (1-5) and "reasoning" (brief explanation) as the value."""

    client = openai.OpenAI()
    evaluation = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": judge_prompt}],
        response_format={"type": "json_object"},
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "500"))
    )
    return json.loads(evaluation.choices[0].message.content)

# Test with a McKinsey consulting scenario
test_question = "How should a mid-size retailer approach digital transformation to improve margins?"
test_response = """To improve margins through digital transformation, a mid-size retailer should focus on three key levers:

1. **Customer Analytics & Personalization**: Implement a CDP (Customer Data Platform) to unify purchase history, browsing behavior, and demographics. Use predictive analytics to personalize promotions, reducing blanket discounting by 15-20% while maintaining conversion rates.

2. **Supply Chain Digitization**: Deploy demand forecasting models to reduce inventory carrying costs by 10-15%. Integrate real-time POS data with procurement to minimize stockouts and overstock situations.

3. **Omnichannel Integration**: Unify online and offline channels through BOPIS (Buy Online, Pick Up In Store) and ship-from-store capabilities. This typically improves inventory turns by 20-30% and increases customer lifetime value.

Implementation should follow a phased 18-month roadmap, starting with quick wins in analytics (months 1-6), then supply chain (months 6-12), and finally full omnichannel (months 12-18). Expected margin improvement: 200-400 basis points."""

result = llm_judge(test_question, test_response, evaluation_criteria)
print(json.dumps(result, indent=2))

---
## Demo 3: Benchmarking Response Quality Across Models

Now we benchmark multiple model configurations against the same set of prompts, collecting metrics like response length, latency, and token usage.

In [ ]:
# Benchmark different configurations on McKinsey consulting topics
test_prompts = [
    "What are the key value creation levers in a post-merger integration?",
    "How should a CPG company respond to private label competition?",
    "What framework would you use to evaluate market entry into Southeast Asia?"
]

configurations = [
    {"model": "gpt-4o-mini", "temperature": 0.0, "label": "GPT-4o-mini (T=0)"},
    {"model": "gpt-4o-mini", "temperature": 0.7, "label": "GPT-4o-mini (T=0.7)"},
    {"model": "gpt-4o-mini", "temperature": 1.0, "label": "GPT-4o-mini (T=1.0)"},
]

results = []
for config in configurations:
    for prompt in test_prompts:
        start_time = time.time()
        response = client.chat.completions.create(
            model=config["model"],
            temperature=config["temperature"],
            messages=[{"role": "user", "content": prompt}],
            max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
        )
        elapsed = time.time() - start_time
        content = response.choices[0].message.content
        results.append({
            "config": config["label"],
            "prompt": prompt[:50] + "...",
            "response_length": len(content),
            "latency_ms": round(elapsed * 1000),
            "tokens_used": response.usage.total_tokens
        })

import pandas as pd
df = pd.DataFrame(results)
print(df.to_string(index=False))

---
## Demo 4: Latency and Cost Analysis

Understanding the cost and latency implications of different models is critical for production deployments. This demo provides utility functions for cost estimation and performance analysis.

In [ ]:
# Cost estimation function
PRICING = {
    "gpt-4o-mini": {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000},
    "gpt-4o": {"input": 2.50 / 1_000_000, "output": 10.00 / 1_000_000},
}

def estimate_cost(model, input_tokens, output_tokens):
    if model in PRICING:
        input_cost = input_tokens * PRICING[model]["input"]
        output_cost = output_tokens * PRICING[model]["output"]
        return round(input_cost + output_cost, 6)
    return None

def analyze_performance(results_df):
    summary = results_df.groupby("config").agg({
        "latency_ms": ["mean", "min", "max"],
        "response_length": "mean",
        "tokens_used": "sum"
    }).round(2)
    print("Performance Summary:")
    print(summary)
    return summary

if len(results) > 0:
    analyze_performance(df)
    
    # Cost estimation example
    for model_name, prices in PRICING.items():
        cost = estimate_cost(model_name, 500, 300)
        print(f"\n{model_name}: Estimated cost for 500 input + 300 output tokens = ${cost}")

---
## Demo 5: A/B Testing Framework

A/B testing lets us rigorously compare two model configurations by running them side-by-side on the same prompts and evaluating both with the LLM judge.

In [ ]:
import random

class ABTestFramework:
    def __init__(self, config_a, config_b):
        self.config_a = config_a
        self.config_b = config_b
        self.results_a = []
        self.results_b = []
        self.client = openai.OpenAI()
    
    def run_test(self, prompts, judge_criteria):
        for prompt in prompts:
            # Get responses from both configurations
            resp_a = self.client.chat.completions.create(
                model=self.config_a["model"],
                temperature=self.config_a.get("temperature", 0.7),
                messages=[
                    {"role": "system", "content": self.config_a.get("system", "You are a McKinsey senior consultant. Provide structured, MECE, data-driven strategic analysis.")},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
            )
            resp_b = self.client.chat.completions.create(
                model=self.config_b["model"],
                temperature=self.config_b.get("temperature", 0.7),
                messages=[
                    {"role": "system", "content": self.config_b.get("system", "You are a McKinsey senior consultant. Provide structured, MECE, data-driven strategic analysis.")},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
            )
            
            score_a = llm_judge(prompt, resp_a.choices[0].message.content, judge_criteria)
            score_b = llm_judge(prompt, resp_b.choices[0].message.content, judge_criteria)
            self.results_a.append(score_a)
            self.results_b.append(score_b)
            print(f"Prompt: {prompt[:50]}... | Config A vs B evaluated")
    
    def get_summary(self):
        print(f"\nA/B Test Results:")
        print(f"Config A: {self.config_a.get('label', 'A')}")
        print(f"Config B: {self.config_b.get('label', 'B')}")
        print(f"Tests run: {len(self.results_a)}")

# Demo (small test)
ab_test = ABTestFramework(
    config_a={"model": "gpt-4o-mini", "temperature": 0.0, "label": "Deterministic"},
    config_b={"model": "gpt-4o-mini", "temperature": 0.7, "label": "Creative"}
)
print("A/B Testing Framework initialized!")
print("Run ab_test.run_test(prompts, criteria) to execute tests")

---
## Demo 6: Scikit-learn Metrics — Precision, Recall, F1 for LLM Classification

When LLMs perform classification tasks (sentiment, intent, category), we can evaluate their accuracy using standard scikit-learn metrics. This gives us quantitative, reproducible scores beyond subjective quality rubrics.

| Metric | What it measures |
|--------|-----------------|
| **Precision** | Of all items predicted as class X, how many were actually X? |
| **Recall** | Of all actual class X items, how many did we correctly predict? |
| **F1 Score** | Harmonic mean of precision and recall — balances both |

In [ ]:
# Demo 6: Scikit-learn Metrics for LLM Classification Evaluation
# Context: Classifying McKinsey client engagement feedback

client = openai.OpenAI()

# Step 1: Define a labeled test dataset — McKinsey client engagement feedback (ground truth)
test_data = [
    {"text": "The engagement exceeded expectations — insights transformed our strategy.", "true_label": "POSITIVE"},
    {"text": "The team's frameworks were razor-sharp and the final readout was outstanding.", "true_label": "POSITIVE"},
    {"text": "McKinsey brought world-class analytical rigor that accelerated our transformation.", "true_label": "POSITIVE"},
    {"text": "The partner's leadership and team's dedication delivered exceptional value.", "true_label": "POSITIVE"},
    {"text": "Deliverables were late and analysis didn't address core issues.", "true_label": "NEGATIVE"},
    {"text": "The recommendations were impractical — clearly the team didn't understand our industry.", "true_label": "NEGATIVE"},
    {"text": "We spent millions and the output was a deck we could have built internally.", "true_label": "NEGATIVE"},
    {"text": "Communication was poor and the project went over budget with no clear ROI.", "true_label": "NEGATIVE"},
    {"text": "The team was professional but recommendations were generic.", "true_label": "NEUTRAL"},
    {"text": "Solid work overall, though nothing we hadn't considered before.", "true_label": "NEUTRAL"},
    {"text": "The analysis was competent but didn't push our thinking in new directions.", "true_label": "NEUTRAL"},
    {"text": "Reasonable engagement — met the brief but didn't go above and beyond.", "true_label": "NEUTRAL"},
]

# Step 2: Use the LLM to classify each piece of client feedback
true_labels = []
predicted_labels = []

for item in test_data:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "You are a McKinsey client feedback classifier. Classify the sentiment of client engagement feedback. Respond with EXACTLY one word: POSITIVE, NEGATIVE, or NEUTRAL."},
            {"role": "user", "content": f"Classify this client feedback: '{item['text']}'"}
        ],
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "5")),
        temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
    )
    prediction = response.choices[0].message.content.strip().upper()
    true_labels.append(item["true_label"])
    predicted_labels.append(prediction)

# Step 3: Compute scikit-learn metrics
labels = ["POSITIVE", "NEGATIVE", "NEUTRAL"]

print("CLASSIFICATION REPORT — McKinsey Client Feedback")
print("=" * 60)
print(classification_report(true_labels, predicted_labels, labels=labels))

# Step 4: Individual metrics
precision = precision_score(true_labels, predicted_labels, labels=labels, average="weighted")
recall = recall_score(true_labels, predicted_labels, labels=labels, average="weighted")
f1 = f1_score(true_labels, predicted_labels, labels=labels, average="weighted")

print(f"\nWeighted Precision: {precision:.4f}")
print(f"Weighted Recall   : {recall:.4f}")
print(f"Weighted F1 Score : {f1:.4f}")

# Step 5: Confusion Matrix Visualization
cm = confusion_matrix(true_labels, predicted_labels, labels=labels)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("Confusion Matrix — McKinsey Client Feedback Classification")

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14,
                color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.colorbar(im)
plt.tight_layout()
plt.show()

print("\nObservation: The confusion matrix shows where the LLM agrees/disagrees with ground truth.")
print("Diagonal = correct predictions. Off-diagonal = misclassifications.")

---
## Demo 7: Using DeepEval and G-Eval for LLM Scoring

[DeepEval](https://github.com/confident-ai/deepeval) is an open-source LLM evaluation framework that implements research-backed metrics, including **G-Eval** (from the paper *"G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment"*).

G-Eval uses an LLM with chain-of-thought to evaluate text quality on criteria like coherence, fluency, relevance, and consistency — achieving higher correlation with human judgments than traditional metrics.

```
pip install deepeval
```

In [ ]:
# Demo 7: Using DeepEval and G-Eval for LLM Scoring

# Install: pip install deepeval
from deepeval import evaluate
from deepeval.metrics import GEval, AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# --- Part A: G-Eval — Custom criteria evaluation ---
print("PART A: G-Eval — Evaluating McKinsey consulting output quality")
print("=" * 60)

# Define G-Eval metrics for consulting quality dimensions
coherence_metric = GEval(
    name="Coherence",
    criteria="Coherence — the response should be logically organized with a clear storyline, following a structured consulting framework (e.g., situation-complication-resolution).",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")
)

fluency_metric = GEval(
    name="Fluency",
    criteria="Fluency — the response should be grammatically correct, well-written, and ready for inclusion in a client-facing consulting deliverable.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")
)

relevance_metric = GEval(
    name="Relevance",
    criteria="Relevance — the response should directly address the client's strategic question and stay on topic throughout, providing actionable consulting insights.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")
)

# Generate a test response using the LLM on a McKinsey topic
client = openai.OpenAI()
test_question = "Explain the benefits of a hub-and-spoke operating model for a global manufacturer."
test_response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": test_question}],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
).choices[0].message.content

print(f"Question: {test_question}")
print(f"Response: {test_response[:200]}...\n")

# Create a test case
test_case = LLMTestCase(
    input=test_question,
    actual_output=test_response
)

# Evaluate with each G-Eval metric
for metric in [coherence_metric, fluency_metric, relevance_metric]:
    metric.measure(test_case)
    print(f"  {metric.name}: {metric.score:.2f} (reason: {metric.reason[:80]}...)")

# --- Part B: Answer Relevancy Metric ---
print("\n" + "=" * 60)
print("PART B: Answer Relevancy Metric — Post-Merger Integration")
print("=" * 60)

relevancy_metric = AnswerRelevancyMetric(
    threshold=0.7,
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")
)

test_case_relevancy = LLMTestCase(
    input="What are best practices for post-merger integration in financial services?",
    actual_output="Post-merger integration in financial services should prioritize four areas: (1) Day-1 readiness for regulatory compliance and customer continuity, (2) technology platform consolidation with a focus on core banking systems, (3) cultural integration through joint leadership workshops and unified incentive structures, and (4) synergy capture tracking with a dedicated PMI office reporting to the CEO. Quick wins in cost synergies (branch rationalization, vendor consolidation) should be pursued in the first 100 days while longer-term revenue synergies (cross-selling, product rationalization) are planned for months 6-18."
)

relevancy_metric.measure(test_case_relevancy)
print(f"Relevancy Score: {relevancy_metric.score:.2f}")
print(f"Passed threshold (0.7): {relevancy_metric.is_successful()}")
print(f"Reason: {relevancy_metric.reason}")

# --- Part C: Comparing multiple responses with DeepEval ---
print("\n" + "=" * 60)
print("PART C: Comparing consulting responses across temperature settings")
print("=" * 60)

question = "What are the key considerations for a private equity portfolio company's digital transformation?"
comparison_results = []

for temp in [0.0, 0.5, 1.0]:
    resp = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": question}],
        temperature=temp,
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
    ).choices[0].message.content

    tc = LLMTestCase(input=question, actual_output=resp)

    coherence_metric.measure(tc)
    relevance_metric.measure(tc)

    comparison_results.append({
        "temperature": temp,
        "coherence": coherence_metric.score,
        "relevance": relevance_metric.score,
        "response_length": len(resp)
    })
    print(f"  T={temp}: coherence={coherence_metric.score:.2f}, relevance={relevance_metric.score:.2f}")

results_df = pd.DataFrame(comparison_results)
print(f"\n{results_df.to_string(index=False)}")

---
## Task 1: Create a Custom Evaluation Rubric for McKinsey Consulting AI

Create a specialized evaluation rubric tailored for agentic AI responses in a McKinsey consulting context, covering behaviors like tool usage accuracy, reasoning quality, MECE decomposition, client readiness, and recommendation quality.

In [ ]:
# TODO: Create a specialized evaluation rubric for McKinsey consulting AI responses that includes:
# 1. At least 5 criteria specific to consulting agentic behaviors:
#    - tool_usage_accuracy: Did the AI use the right analytical tools/frameworks (e.g., market sizing, benchmarking)?
#    - reasoning_quality: How rigorous and logical is the analytical reasoning?
#    - mece_decomposition: Does the response break down the problem in a MECE (Mutually Exclusive, Collectively Exhaustive) way?
#    - client_readiness: Is the output polished enough for a client-facing deliverable?
#    - recommendation_quality: Are the recommendations specific, prioritized, and backed by evidence?
# 2. A scoring scale (1-5) with descriptions for each level tailored to consulting standards
# 3. A function that displays the rubric in a formatted way
#
# Hint: Think about what makes a CONSULTING AI agent response different from a generic chatbot response
# Hint: McKinsey consultants use frameworks, structured thinking, and data-driven insights
# Hint: Use a nested dictionary structure: {criterion: {score: description}}

def create_agentic_rubric():
    # YOUR CODE HERE
    pass

# Display the rubric
# create_agentic_rubric()

---
## Task 2: Compare Multiple Models on McKinsey Consulting Tasks

Build a function that systematically compares model configurations by running McKinsey-style consulting prompts, scoring with LLM-as-a-Judge, and organizing results into a DataFrame.

In [ ]:
# TODO: Create a model comparison function that:
# 1. Takes a list of McKinsey consulting test prompts and model configurations
# 2. Runs each prompt through each model configuration
# 3. Evaluates responses using llm_judge (from Demo 2)
# 4. Returns a pandas DataFrame with all scores organized by model and criteria
#
# Hint: Use nested loops — outer for configs, inner for prompts
# Hint: Store results as a list of dicts, then convert to DataFrame
# Hint: Include latency and token count alongside quality scores

def compare_models(prompts, configs, criteria):
    # YOUR CODE HERE
    pass

# test_prompts = [
#     "What are the key synergies in a healthcare M&A transaction?",
#     "How should a bank modernize its core technology platform?"
# ]
# compare_models(test_prompts, configurations, evaluation_criteria)

---
## Summary

In this session, we covered:

1. **Evaluation Rubrics** - Defining structured criteria and scoring scales for consistent assessment
2. **LLM-as-a-Judge** - Using AI to automate evaluation with structured JSON scoring
3. **Benchmarking** - Comparing model configurations across standardized test prompts
4. **Cost and Latency Analysis** - Understanding the practical trade-offs of different models
5. **A/B Testing** - Rigorous side-by-side comparison of model configurations
6. **Scikit-learn Metrics** - Using precision, recall, F1 score, and confusion matrices to quantitatively evaluate LLM classification tasks against ground truth labels
7. **DeepEval and G-Eval** - Leveraging research-backed LLM evaluation frameworks for automated scoring on coherence, fluency, relevance, and custom criteria

### Key Takeaways
- Always define clear evaluation criteria before comparing models
- Automated evaluation with LLM judges scales better than manual review
- For classification tasks, scikit-learn metrics (precision, recall, F1) give hard quantitative numbers
- DeepEval's G-Eval metric uses chain-of-thought LLM judges that correlate better with human judgments
- Consider both quality AND cost/latency when selecting a model for production
- A/B testing provides statistical confidence in model selection decisions
- Visualization helps communicate evaluation results to stakeholders

In [ ]:
# TODO: Build an EvaluationPipeline class that:
# 1. Takes a dataset of McKinsey consulting (question, expected_behavior) pairs
# 2. Runs the model on each question
# 3. Evaluates using LLM-as-a-judge with McKinsey consulting criteria
# 4. Aggregates scores and identifies weak areas
# 5. Generates a summary report
#
# Test cases to use:
#   ("What are the key risks in a cross-border acquisition?", "Should cover regulatory, cultural, financial, and operational risks in a MECE structure")
#   ("How would you size the market for electric vehicle charging in India?", "Should use a top-down or bottom-up approach with clear assumptions and data sources")
#   ("What operating model changes should a telco make to improve EBITDA margins?", "Should identify cost levers, organizational changes, and digital enablement opportunities")
#
# Hint: Store results in a list, then use pandas for aggregation
# Hint: Weak areas = criteria with average score below 3.5
# Hint: The report should include per-criterion averages and overall score

class EvaluationPipeline:
    def __init__(self, model_config, criteria):
        # YOUR CODE HERE
        pass
    
    def run(self, test_cases):
        # YOUR CODE HERE
        pass
    
    def generate_report(self):
        # YOUR CODE HERE
        pass

---
## Task 4: Analyze and Visualize McKinsey Consulting Evaluation Results

Create visualization functions to plot bar charts, radar charts, and latency-vs-quality scatter plots for comparing evaluation results across model configurations on McKinsey consulting tasks.

In [ ]:
# TODO: Create visualization functions that:
# 1. Plot a bar chart comparing average scores across McKinsey consulting criteria
#    (strategic_relevance, analytical_rigor, mece_structure, actionability, executive_readiness)
# 2. Plot a radar/spider chart for multi-dimensional comparison of consulting quality
# 3. Plot latency vs quality scatter plot to find the optimal model for consulting use cases
# 4. Generate a summary markdown table for engagement leadership review
#
# Hint: Use matplotlib for plotting
# Hint: For radar charts, use polar coordinates: ax = plt.subplot(111, polar=True)
# Hint: Add labels, titles, and legends for clarity
# Hint: Consider adding a "quality threshold" line at score 4.0 (McKinsey standard)

import matplotlib.pyplot as plt
import numpy as np

def plot_criteria_comparison(results_df):
    # YOUR CODE HERE
    pass

def plot_radar_chart(scores_dict, title="McKinsey Consulting AI — Model Comparison"):
    # YOUR CODE HERE
    pass

def plot_latency_vs_quality(results_df):
    # YOUR CODE HERE
    pass

# Generate visualizations with sample data or results from Task 2

---
## Summary

In this session, we covered:

1. **Evaluation Rubrics** - Defining structured criteria and scoring scales for consistent assessment
2. **LLM-as-a-Judge** - Using AI to automate evaluation with structured JSON scoring
3. **Benchmarking** - Comparing model configurations across standardized test prompts
4. **Cost and Latency Analysis** - Understanding the practical trade-offs of different models
5. **A/B Testing** - Rigorous side-by-side comparison of model configurations

### Key Takeaways
- Always define clear evaluation criteria before comparing models
- Automated evaluation with LLM judges scales better than manual review
- Consider both quality AND cost/latency when selecting a model for production
- A/B testing provides statistical confidence in model selection decisions
- Visualization helps communicate evaluation results to stakeholders